In [0]:
#look first (no need to send me the output unless cell 2 fails)
C = "capstone_supply_chain"
b = spark.table(f"{C}.bronze.raw_shipping_logistics")
b.select("shipment_id", "mode", "status", "weight_kg", "freight_cost_usd",
         "origin.departure", "destination.eta").show(truncate=False)

In [0]:
#build two tables, silver.shipments and silver.shipment_events
from pyspark.sql import functions as F
from pyspark.sql.window import Window

C = "capstone_supply_chain"
b = spark.table(f"{C}.bronze.raw_shipping_logistics")

# One row per tracking event (the nested list becomes rows)
events = (b
    .select("shipment_id", F.posexplode_outer("tracking_events").alias("event_pos", "ev"))
    .select(
        "shipment_id",
        (F.col("event_pos") + 1).alias("event_seq"),
        F.col("ev.event").alias("event"),
        F.col("ev.location").alias("location"),
        F.expr("try_cast(ev.`timestamp` AS TIMESTAMP)").alias("event_ts"),
    )
    .filter(F.col("event").isNotNull())
)
events.write.mode("overwrite").saveAsTable(f"{C}.silver.shipment_events")

# Latest event and event count per shipment
w = Window.partitionBy("shipment_id").orderBy(F.col("event_ts").desc_nulls_last(), F.col("event_seq").desc())
last_event = (events.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("shipment_id",
            F.col("event").alias("last_event"),
            F.col("location").alias("last_event_location"),
            F.col("event_ts").alias("last_event_ts")))
event_counts = events.groupBy("shipment_id").agg(F.count("*").alias("event_count"))

# One row per shipment, with the nested structures flattened
shipments = (b.select(
        "shipment_id",
        F.col("po_reference").alias("po_id"),
        "awb", "pro_number", "carrier",
        F.trim(F.col("mode")).alias("mode"),
        F.initcap(F.trim(F.col("status"))).alias("status"),
        F.col("origin.city").alias("origin_city"),
        F.col("origin.country").alias("origin_country"),
        F.coalesce(F.col("origin.airport"), F.col("origin.port")).alias("origin_hub"),
        F.expr("try_cast(origin.departure AS TIMESTAMP)").alias("departure_ts"),
        F.col("destination.city").alias("destination_city"),
        F.col("destination.country").alias("destination_country"),
        F.coalesce(F.col("destination.airport"), F.col("destination.port")).alias("destination_hub"),
        F.expr("try_cast(destination.eta AS TIMESTAMP)").alias("eta_ts"),
        F.col("weight_kg"),
        F.col("freight_cost_usd"),
        F.col("container.number").alias("container_number"),
        F.col("container.seal").alias("container_seal"),
        F.col("container.type").alias("container_type"),
        F.col("customs.broker").alias("customs_broker"),
        F.col("customs.status").alias("customs_status"),
        F.col("customs.hs_code").alias("hs_code"),
        F.col("customs.duty_paid_cad").alias("duty_paid_cad"),
        F.col("customs.usmca_qualified").alias("usmca_qualified"),
    )
    .withColumn("transit_days", F.datediff(F.to_date("eta_ts"), F.to_date("departure_ts")))
    .withColumn("freight_cost_per_kg",
        F.when(F.col("weight_kg") > 0, F.round(F.col("freight_cost_usd") / F.col("weight_kg"), 2)))
    .join(event_counts, "shipment_id", "left")
    .join(last_event, "shipment_id", "left")
    .withColumn("silver_processed_at", F.current_timestamp())
    .filter(F.col("shipment_id").isNotNull())
    .dropDuplicates(["shipment_id"])
)
shipments.write.mode("overwrite").saveAsTable(f"{C}.silver.shipments")

print("✅ silver.shipments:", shipments.count(), " silver.shipment_events:", events.count())

In [0]:
#checks
sh = spark.table(f"{C}.silver.shipments")
ev = spark.table(f"{C}.silver.shipment_events")

print("shipments:", sh.count(), " distinct ids:", sh.select("shipment_id").distinct().count())
print("events in silver:", ev.count(), " events in bronze:",
      b.select(F.sum(F.size("tracking_events"))).first()[0])

# Nothing lost, and dates parsed
sh.select(
    F.sum(F.col("departure_ts").isNull().cast("int")).alias("null_departure"),
    F.sum(F.col("eta_ts").isNull().cast("int")).alias("null_eta"),
    F.sum((F.col("transit_days") < 0).cast("int")).alias("negative_transit"),
).show()

# Does every shipment point at a real purchase order?
po = spark.table(f"{C}.silver.purchase_orders").select("po_id")
print("shipments with an unknown PO:", sh.join(po, "po_id", "left_anti").count())

sh.select("shipment_id", "po_id", "mode", "status", "origin_city", "destination_city",
          "transit_days", "event_count", "last_event", "freight_cost_per_kg").show(truncate=False)

sh.select("shipment_id", "mode", "origin_city", "origin_hub", "origin_country",
          "destination_city", "destination_hub", "destination_country",
          "freight_cost_usd", "weight_kg").show(truncate=False)